# Evaluation of Summarization Metrics using DUC 2004 Dataset

> **Research Notebook** | March 2026
>
> Evaluates ROUGE, BERTScore, and BARTScore on a DUC 2004-style dataset.
> Covers all four phases: Dataset → Evaluation → Analysis → Innovation.

---

**Table of Contents**
1. Setup & Imports
2. Phase 1 — Dataset Loading
3. Phase 2 — Metric Computation
4. Phase 3 — Correlation & Visualization
5. Phase 4 — Innovation (Hybrid Metric, Error Analysis, Regression)
6. Conclusions

## 1. Setup & Imports

In [ ]:
import sys
import warnings
from pathlib import Path

warnings.filterwarnings('ignore')

# Ensure project root is on path
ROOT = Path('..').resolve()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import pandas as pd
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display, Image

sns.set_theme(style='whitegrid', font_scale=1.1)
print('Setup complete. Root:', ROOT)

## 2. Phase 1 — Dataset Loading

In [ ]:
from data.duc_simulate import generate_dataset
from src.loader import load_dataset, get_pairs, describe_dataset

DATASET_PATH = '../data/duc_2004_simulated.json'

# Generate if needed
generate_dataset(output_path=DATASET_PATH)

# Load
data = load_dataset(DATASET_PATH)
ids, references, systems = get_pairs(data)
describe_dataset(data)

In [ ]:
# Preview dataset
pd.DataFrame(data)[['id', 'reference', 'system']].head(5)

### Sample Analysis
Let's look at a few interesting samples to understand the variety:

In [ ]:
# Show paraphrase case (low ROUGE expected, high BERTScore expected)
sample_11 = next(s for s in data if s['id'] == 'DUC2004_011')
print('=== Paraphrase Case (DUC2004_011) ===')
print(f"Reference : {sample_11['reference']}")
print(f"System    : {sample_11['system']}")
print()

# Show hallucinated case (low both metrics expected)
sample_13 = next(s for s in data if s['id'] == 'DUC2004_013')
print('=== Hallucinated Summary (DUC2004_013) ===')
print(f"Reference : {sample_13['reference']}")
print(f"System    : {sample_13['system']}")

## 3. Phase 2 — Metric Computation

We compute ROUGE, BERTScore, and (optionally) BARTScore.  
**Tip:** Set `USE_BARTSCORE = False` for fast testing without downloading the 1.6GB BART model.

In [ ]:
from src.metrics import compute_rouge, compute_bertscore, compute_bartscore

USE_BARTSCORE = False  # Set to True if you want BARTScore (requires ~1.6GB model)
BERTSCORE_MODEL = 'distilbert-base-uncased'  # or 'roberta-large' for better accuracy

print(f'Settings: use_bartscore={USE_BARTSCORE}, bertscore_model={BERTSCORE_MODEL}')

In [ ]:
# --- ROUGE ---
print('Computing ROUGE...')
rouge_scores = compute_rouge(systems, references)
print('ROUGE computed:', {k: round(sum(v)/len(v), 4) for k, v in rouge_scores.items()})

In [ ]:
# --- BERTScore ---
print('Computing BERTScore...')
bert_scores = compute_bertscore(systems, references, model_type=BERTSCORE_MODEL)
print('BERTScore F1 mean:', round(sum(bert_scores['bertscore_f1'])/len(bert_scores['bertscore_f1']), 4))

In [ ]:
# --- BARTScore (optional) ---
bart_scores = {}
if USE_BARTSCORE:
    print('Computing BARTScore (this may take a few minutes)...')
    bart_scores = compute_bartscore(systems, references)
    print('BARTScore mean:', round(sum(bart_scores['bartscore'])/len(bart_scores['bartscore']), 4))
else:
    print('BARTScore skipped (USE_BARTSCORE=False).')

In [ ]:
# Build results DataFrame
records = {'id': ids, 'reference': references, 'system': systems}
records.update(rouge_scores)
records.update(bert_scores)
if bart_scores:
    records.update(bart_scores)

df = pd.DataFrame(records)
df.to_csv('../results/evaluation_results.csv', index=False)
print(f'Results saved. Shape: {df.shape}')
df.head()

### Score Distribution Preview

In [ ]:
metric_cols = [c for c in df.columns if c not in ('id', 'reference', 'system')]
df[metric_cols].describe().round(4)

## 4. Phase 3 — Correlation & Visualization

In [ ]:
from src.analysis import (
    compute_correlations, print_correlations,
    plot_correlation_heatmap, plot_distributions,
    plot_rouge_vs_bertscore, plot_scatter_matrix
)
from pathlib import Path

Path('../results').mkdir(exist_ok=True)

In [ ]:
# Correlation matrices
corr = compute_correlations(df)
print_correlations(corr)

In [ ]:
# Correlation heatmap
heatmap_path = plot_correlation_heatmap(df, output_dir='../results/', method='pearson')
Image(heatmap_path)

In [ ]:
# Distribution plots
dist_path = plot_distributions(df, output_dir='../results/')
Image(dist_path)

In [ ]:
# ROUGE vs BERTScore comparison
rv_path = plot_rouge_vs_bertscore(df, output_dir='../results/')
Image(rv_path)

### Key Observations

- **ROUGE-1 and ROUGE-2** are strongly correlated (both lexical overlap metrics)
- **BERTScore-F1** shows moderate correlation with ROUGE on semantic matches but diverges on paraphrase cases
- **Samples 011/012** (paraphrase) show high BERTScore but lower-than-expected ROUGE
- **Samples 013/014** (hallucination) show distinctly low scores across all metrics

## 5. Phase 4 — Innovation

### Innovation A: Hybrid Metric

In [ ]:
from src.innovation import compute_hybrid_metric, plot_hybrid_comparison

df_hybrid = compute_hybrid_metric(df, alpha=0.25, beta=0.15, gamma=0.60)

print('Hybrid Score stats:')
print(df_hybrid['hybrid_score'].describe().round(4))

hybrid_path = plot_hybrid_comparison(df_hybrid, output_dir='../results/')
Image(hybrid_path)

### Innovation B: Error Analysis

In [ ]:
from src.innovation import error_analysis

df_err = error_analysis(df_hybrid, output_dir='../results/')

# Show samples categorized as 'Semantic Match' — where ROUGE fails
semantic_matches = df_err[df_err['disagreement_category'].str.contains('Semantic', na=False)]
print(f"\nSamples where ROUGE fails (Semantic Match): {len(semantic_matches)}")
display(semantic_matches[['id', 'rouge1', 'bertscore_f1', 'score_delta', 'disagreement_category']])

Image('../results/disagreement_analysis.png')

### Innovation C: Regression Model

In [ ]:
from src.innovation import train_regression_model

reg_results = train_regression_model(df_err, target_col='hybrid_score', output_dir='../results/')

print(f"\nLOO R²  : {reg_results['loo_r2']}")
print(f"LOO MSE : {reg_results['loo_mse']}")
print(f"\nFeature Importances:")
for feat, coef in reg_results['feature_importances'].items():
    print(f"  {feat:<30} {coef:+.4f}")

In [ ]:
Image('../results/feature_importance.png')

In [ ]:
Image('../results/predicted_vs_actual.png')

## 6. Conclusions

### ROUGE vs BERTScore
| Aspect | ROUGE | BERTScore |
|---|---|---|
| Paraphrase robustness | ❌ Poor | ✅ Good |
| Speed | ✅ Very fast | ⚠️ Slow (model inference) |
| Interpretability | ✅ High | ⚠️ Less transparent |
| Requires reference | ✅ Yes | ✅ Yes |
| Human correlation | ⚠️ Moderate | ✅ Higher |

### Key Findings
1. **ROUGE fails on paraphrasing** (DUC2004_011, _012) — semantic meaning preserved but lexical overlap is low
2. **BERTScore** correctly identifies semantic matches that ROUGE misses
3. **Hybrid metric** balances recall of lexical overlap (ROUGE) with semantic understanding (BERTScore)
4. **Regression model** demonstrates that metric combinations can predict composite quality better than any single metric

### Recommendations
- Use **BERTScore-F1** as primary metric in research settings
- Use **ROUGE-1+2** for fast baseline comparisons
- Use **Hybrid Score** for production evaluation systems
- Always run **error analysis** to understand where metrics disagree before drawing conclusions